# Patient extraction · top/bottom-decile transferred-XGBoost risk patients and their pathology reports

Data request (Elio). At each landmark (ADT +0 / +90 / +180 days) the platinum-trained federated XGBoost model, refit
**without Dana-Farber** and scored on the local DFCI ADT cohort in notebook 04, gives each patient a risk score. This
notebook takes the patients in the **top or bottom decile** of that score at **any** landmark and writes:

| File | Content |
|---|---|
| `patient_manifest.csv` | one row per patient: MRN, ADT start, risk score / rank percentile / top-or-bottom-decile flag at each landmark, AVPC / platinum / NEPC event flag and days from ADT start, number of pathology notes |
| `pathology_reports.tsv` | one row per pathology report (all dates): every manifest column plus the note metadata (`RPT_ID`, `EVENT_DATE`, `PROC_DESC`, `RPT_TYPE`, `FILE`, days from ADT start) and `RPT_TEXT` |

Output folder: `<COMPASS data root>/elio_data_requests/high_vs_low_risk_transfer_pxs/`. **Both files contain PHI**
(MRNs, report text); they stay under the protected data root and are never copied into the repository.

Run notebook 04 first: this reads its protected `all_models/transfer_scores.csv`. The risk score does not depend on the
evaluated endpoint, so the platinum rows are used. Deciles are taken among patients with finite numeric scores at that landmark.
Missing, nonnumeric, or infinite scores are excluded from cutoffs and ranks and labeled `not scored` in the manifest.
A patient is flagged `top` when their score is at or above the landmark's 90th percentile and `bottom` when at or below
its 10th. Ties at a cutoff are included, so groups may exceed 10%. Extraction stops if a landmark has no valid scores
or its two cutoffs coincide, because that landmark cannot separate high- and low-risk groups.

## Configuration

In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp

PROJECT_ROOT = cp.PROJECT_ROOT
for _p in (str(PROJECT_ROOT),):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd

from survival_common import federated_endpoint_transfer as fet
from survival_common import federated_scoring as fs

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
SCORES_PATH = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"   # notebook 04
DATA_PATH   = DATA_ROOT / "longitudinal_prediction_data_adt.csv"                                      # ADT anchor + endpoints
PATHOLOGY_PATH = Path(os.environ.get(
    "PROFILE_PATHOLOGY_NOTES",
    "/data/gusev/USERS/jpconnor/data/PROFILE_DATA/CLINICAL_NOTES/PATHOLOGY_NOTES.parquet"))
OUTPUT_DIR = DATA_ROOT / "elio_data_requests" / "high_vs_low_risk_transfer_pxs"

MODEL_FAMILY = "xgboost_cox"
SCORE_ENDPOINT = "platinum"        # the model is platinum-trained; risk_score is identical across endpoints
LANDMARKS = (0, 90, 180)
DECILE = 0.10                      # top and bottom 10% at each landmark
ENDPOINTS = ("avpc", "platinum", "nepc")

## Risk scores and deciles

One XGBoost full-model score per patient and landmark. The age-only baseline and any other family are dropped, and the
selection must leave exactly one row per patient and landmark.

In [ ]:
scores = pd.read_csv(SCORES_PATH, low_memory=False)
print(scores.groupby(["model_family", "config"]).size())

sel = scores[(scores["model_family"] == MODEL_FAMILY) & (scores["config"] != "baseline")
             & (scores["endpoint"] == SCORE_ENDPOINT) & (scores["landmark_days"].isin(LANDMARKS))]
sel = sel[[fs.ID_COL, "landmark_days", "risk_score"]]
dup = sel.duplicated([fs.ID_COL, "landmark_days"], keep=False)
assert not dup.any(), f"{dup.sum()} rows: more than one {MODEL_FAMILY} model/arm per patient and landmark; narrow the filter above"
assert set(sel["landmark_days"]) == set(LANDMARKS), f"landmarks found: {sorted(sel['landmark_days'].unique())}"

def decile_flags(g):
    if not 0 < DECILE < 0.5:
        raise ValueError("DECILE must be between 0 and 0.5 (exclusive).")
    g = g.copy()
    risk = pd.to_numeric(g["risk_score"], errors="coerce").astype(float)
    risk = risk.where(np.isfinite(risk))
    valid = risk.notna()
    landmark = g["landmark_days"].iloc[0]
    if not valid.any():
        raise ValueError(f"Landmark {landmark}: no finite numeric risk scores; extraction stopped.")
    lo, hi = risk[valid].quantile([DECILE, 1 - DECILE])
    if lo >= hi:
        raise ValueError(
            f"Landmark {landmark}: bottom and top cutoffs coincide; "
            "cannot separate high- and low-risk groups. Extraction stopped."
        )
    g["risk_score"] = risk
    g["risk_percentile"] = risk.rank(pct=True, method="average")
    g["decile_group"] = np.select(
        [risk >= hi, risk <= lo, valid], ["top", "bottom", "middle"], default="not scored"
    )
    g.attrs.update(lo=lo, hi=hi)
    return g

by_lm = {lm: decile_flags(g) for lm, g in sel.groupby("landmark_days")}
cutoffs = pd.DataFrame({lm: {"n_scored": int(g["risk_score"].notna().sum()),
                            "n_not_scored": int(g["risk_score"].isna().sum()),
                            "bottom_cutoff": g.attrs["lo"], "top_cutoff": g.attrs["hi"],
                            "n_top": int((g["decile_group"] == "top").sum()),
                            "n_bottom": int((g["decile_group"] == "bottom").sum())} for lm, g in by_lm.items()}).T
cutoffs

In [ ]:
wide = None
for lm, g in by_lm.items():
    part = g.rename(columns={"risk_score": f"risk_score_lm{lm}", "risk_percentile": f"risk_percentile_lm{lm}",
                             "decile_group": f"decile_group_lm{lm}"}).drop(columns="landmark_days")
    wide = part if wide is None else wide.merge(part, on=fs.ID_COL, how="outer")

group_cols = [f"decile_group_lm{lm}" for lm in LANDMARKS]
extreme = wide[group_cols].isin(["top", "bottom"]).any(axis=1)
cohort = wide[extreme].rename(columns={fs.ID_COL: "DFCI_MRN"}).copy()
for lm in LANDMARKS:
    col = f"decile_group_lm{lm}"
    cohort[f"top_decile_lm{lm}"] = cohort[col].eq("top")
    cohort[f"bottom_decile_lm{lm}"] = cohort[col].eq("bottom")
    cohort[col] = cohort[col].fillna("not scored")
print(f"{len(cohort):,} patients top/bottom decile at >=1 landmark (of {len(wide):,} patients in the score file)")

## ADT start and endpoints

ADT start is `TREATMENT_ANCHOR_DATE`, the anchor the landmarks are measured from. Each endpoint's date is used only when
its indicator is 1 (the date columns are back-filled for non-events upstream). `days_to_<endpoint>` counts from ADT start
and can be negative if the event was dated before ADT.

In [ ]:
local = fet.read_local_frame(DATA_PATH, ENDPOINTS)
cols = [fet.LOCAL_ID_COL, "TREATMENT_ANCHOR_DATE"] + [c for ep in ENDPOINTS for c in fet.ENDPOINT_SOURCES[ep]]
pt = local[cols].drop_duplicates(fet.LOCAL_ID_COL).copy()
pt["adt_start_datetime"] = pd.to_datetime(pt["TREATMENT_ANCHOR_DATE"], errors="coerce", format="mixed")
for ep in ENDPOINTS:
    ind, dt = fet.ENDPOINT_SOURCES[ep]
    event = pd.to_numeric(pt[ind], errors="coerce").fillna(0).eq(1)
    date = pd.to_datetime(pt[dt], errors="coerce", format="mixed").where(event)
    pt[f"{ep}_event"] = event
    pt[f"{ep}_event_date"] = date
    pt[f"days_adt_to_{ep}"] = (date - pt["adt_start_datetime"]).dt.days
keep = [fet.LOCAL_ID_COL, "adt_start_datetime"] + [c for ep in ENDPOINTS for c in
        (f"{ep}_event", f"{ep}_event_date", f"days_adt_to_{ep}")]
pt = pt[keep].rename(columns={fet.LOCAL_ID_COL: "DFCI_MRN"})
pt["DFCI_MRN"] = pd.to_numeric(pt["DFCI_MRN"], errors="coerce").astype("Int64")
cohort["DFCI_MRN"] = pd.to_numeric(cohort["DFCI_MRN"], errors="coerce").astype("Int64")

manifest = cohort.merge(pt, on="DFCI_MRN", how="left", validate="one_to_one")
assert manifest["adt_start_datetime"].notna().all(), "scored patients missing an ADT start in the local frame"
manifest[[f"{ep}_event" for ep in ENDPOINTS]].sum()

## Pathology reports

Every pathology report on file for these patients, at any date (including after ADT start and after any event). Report
dates relative to ADT start are given so the request can be restricted downstream.

In [ ]:
import polars as pl

notes = (pl.scan_parquet(PATHOLOGY_PATH)
         .filter(pl.col("DFCI_MRN").cast(pl.Int64, strict=False).is_in(manifest["DFCI_MRN"].dropna().astype("int64").tolist()))
         .collect().to_pandas())
notes["DFCI_MRN"] = pd.to_numeric(notes["DFCI_MRN"], errors="coerce").astype("Int64")
notes["EVENT_DATE"] = pd.to_datetime(notes["EVENT_DATE"], errors="coerce", format="mixed")
print(f"{len(notes):,} pathology reports for {notes['DFCI_MRN'].nunique():,} of {len(manifest):,} patients")

counts = notes.groupby("DFCI_MRN").size().rename("n_pathology_notes")
manifest = manifest.merge(counts, on="DFCI_MRN", how="left")
manifest["n_pathology_notes"] = manifest["n_pathology_notes"].fillna(0).astype(int)
manifest = manifest.sort_values("DFCI_MRN").reset_index(drop=True)

# Column order: identity, ADT start, per-landmark score / percentile / flags, endpoints, note count.
lm_cols = [c for lm in LANDMARKS for c in (f"risk_score_lm{lm}", f"risk_percentile_lm{lm}", f"decile_group_lm{lm}",
                                          f"top_decile_lm{lm}", f"bottom_decile_lm{lm}")]
ep_cols = [c for ep in ENDPOINTS for c in (f"{ep}_event", f"days_adt_to_{ep}", f"{ep}_event_date")]
manifest = manifest[["DFCI_MRN", "adt_start_datetime"] + lm_cols + ep_cols + ["n_pathology_notes"]]
print(f"Manifest ready: {len(manifest):,} patients; "
      f"{manifest['n_pathology_notes'].eq(0).sum():,} without pathology reports")

## Write

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / "patient_manifest.csv"
manifest.to_csv(manifest_path, index=False)

note_cols = [c for c in ("RPT_ID", "EVENT_DATE", "PROC_DESC", "RPT_TYPE", "FILE") if c in notes.columns]
rep = notes[["DFCI_MRN"] + note_cols + ["RPT_TEXT"]].rename(columns={
    "EVENT_DATE": "note_date", **{c: f"note_{c.lower()}" for c in note_cols if c != "EVENT_DATE"}})
rep = manifest.merge(rep, on="DFCI_MRN", how="inner", validate="one_to_many")
rep["note_days_from_adt_start"] = (rep["note_date"] - rep["adt_start_datetime"]).dt.days
text_col = rep.pop("RPT_TEXT").rename("note_text")
rep = pd.concat([rep, text_col], axis=1).sort_values(["DFCI_MRN", "note_date"], kind="stable")
# Report bodies carry tabs and newlines; pandas quotes such fields, which any TSV reader honours.
tsv_path = OUTPUT_DIR / "pathology_reports.tsv"
rep.to_csv(tsv_path, sep="\t", index=False)

check = pd.read_csv(tsv_path, sep="\t", low_memory=False)
assert len(check) == len(rep) == int(manifest["n_pathology_notes"].sum())
print(f"wrote {manifest_path} ({len(manifest):,} patients)")
print(f"wrote {tsv_path} ({len(rep):,} reports; round-trip row count verified)")